In [1]:
# ── Cell 1: GPU check + dependencies ──
import torch
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Install unsloth FIRST (must be before any trl/transformers import)
!pip install -q "unsloth[colab-new]" -U
!pip install -q "trl>=0.12.0" "peft>=0.12.0" "accelerate>=0.30.0" "datasets>=2.18.0" matplotlib
print("Dependencies installed.")


GPU: Tesla T4 | VRAM: 15.6 GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [2]:
# ── Cell 2: Create directory structure ──
import os
for d in ["/content/benchmark",
          "/content/data/tenacious_bench_v0.1/dev",
          "/content/data/tenacious_bench_v0.1/held_out",
          "/content/ablations"]:
    os.makedirs(d, exist_ok=True)
print("Directories created.")


Directories created.


In [4]:
# ── Cell 3: Upload all files ──
from google.colab import files
import shutil

print("Upload these 7 files when prompted (one round):")
print("  1) train_simpo_judge.py")
print("  2) tenacious_judge_act5_pairs.jsonl")
print("  3) scoring_evaluator.py")
print("  4) schema.json")
print("  5) dev_tasks.jsonl")
print("  6) held_tasks.jsonl")
print("  7) run_ablation.py")
print()

uploaded_all = files.upload()

# Move files to correct locations
for fname in uploaded_all:
    if fname == "scoring_evaluator.py" or fname == "schema.json":
        shutil.move(f"/content/{fname}", f"/content/benchmark/{fname}")
        print(f"  {fname} → /content/benchmark/{fname}")
    elif fname == "dev_tasks.jsonl":
        shutil.move(f"/content/{fname}", f"/content/data/tenacious_bench_v0.1/dev/{fname}")
        print(f"  {fname} → /content/data/tenacious_bench_v0.1/dev/{fname}")
    elif fname == "held_tasks.jsonl":
        shutil.move(f"/content/{fname}", f"/content/data/tenacious_bench_v0.1/held_out/{fname}")
        print(f"  {fname} → /content/data/tenacious_bench_v0.1/held_out/{fname}")
    else:
        print(f"  {fname} → /content/{fname}")

print("\nAll files uploaded.")


Upload these 7 files when prompted (one round):
  1) train_simpo_judge.py
  2) tenacious_judge_act5_pairs.jsonl
  3) scoring_evaluator.py
  4) schema.json
  5) dev_tasks.jsonl
  6) held_tasks.jsonl
  7) run_ablation.py



Saving dev_tasks.jsonl to dev_tasks.jsonl
Saving held_tasks.jsonl to held_tasks.jsonl
Saving run_ablation.py to run_ablation.py
Saving schema.json to schema.json
Saving scoring_evaluator.py to scoring_evaluator.py
Saving tenacious_judge_act5_pairs.jsonl to tenacious_judge_act5_pairs.jsonl
Saving train_simpo_judge.py to train_simpo_judge.py
  dev_tasks.jsonl → /content/data/tenacious_bench_v0.1/dev/dev_tasks.jsonl
  held_tasks.jsonl → /content/data/tenacious_bench_v0.1/held_out/held_tasks.jsonl
  run_ablation.py → /content/run_ablation.py
  schema.json → /content/benchmark/schema.json
  scoring_evaluator.py → /content/benchmark/scoring_evaluator.py
  tenacious_judge_act5_pairs.jsonl → /content/tenacious_judge_act5_pairs.jsonl
  train_simpo_judge.py → /content/train_simpo_judge.py

All files uploaded.


In [5]:
import os
required = {
    "/content/train_simpo_judge.py":                               "Training script",
    "/content/tenacious_judge_act5_pairs.jsonl":                     "200 preference pairs",
    "/content/run_ablation.py":                                    "Ablation script",
    "/content/benchmark/scoring_evaluator.py":                     "Deterministic judge",
    "/content/benchmark/schema.json":                              "Schema",
    "/content/data/tenacious_bench_v0.1/dev/dev_tasks.jsonl":      "78-task dev",
    "/content/data/tenacious_bench_v0.1/held_out/held_tasks.jsonl":"65-task held-out",
}
all_ok = True
for path, desc in required.items():
    exists = os.path.exists(path)
    print(f"  [{'OK' if exists else 'MISSING'}] {path}")
    if not exists: all_ok = False
assert all_ok, "Some files missing!"
print("\nAll files verified.")


  [OK] /content/train_simpo_judge.py
  [OK] /content/tenacious_judge_act5_pairs.jsonl
  [OK] /content/run_ablation.py
  [OK] /content/benchmark/scoring_evaluator.py
  [OK] /content/benchmark/schema.json
  [OK] /content/data/tenacious_bench_v0.1/dev/dev_tasks.jsonl
  [OK] /content/data/tenacious_bench_v0.1/held_out/held_tasks.jsonl

All files verified.


In [6]:
from pathlib import Path
import re

CORRECT_MODEL = "unsloth/Qwen2.5-3B-Instruct"

# All model names that might need to be replaced
# This includes previous models and the one from the last successful patch.
OLD_MODELS_TO_REPLACE = [
    "unsloth/Qwen3-30B-A3B-Instruct-2507", # Model from previous successful patch
    "unsloth/Qwen2.5-0.5B-Instruct",
    "unsloth/Qwen2.5-1.5B-Instruct",
    "unsloth/qwen-2.5-0.5b-instruct",
    "unsloth/qwen-2.5-1.5b",
    "unsloth/Qwen2.5-3B-Instruct" # Include the target model itself to handle cases where it might already be present
]

# === Patch train_simpo_judge.py ===
train_path = Path("/content/train_simpo_judge.py")
train_text = train_path.read_text()

# Replace any of the old model names with the new CORRECT_MODEL
for old_model in OLD_MODELS_TO_REPLACE:
    train_text = train_text.replace(old_model, CORRECT_MODEL)

train_path.write_text(train_text)
assert CORRECT_MODEL in train_path.read_text()
print(f"[OK] train_simpo_judge.py -> {CORRECT_MODEL}")

# === Patch run_ablation.py ===
ablation_path = Path("/content/run_ablation.py")
ablation_text = ablation_path.read_text()

# Fix ROOT path for Colab
ablation_text = ablation_text.replace(
    "ROOT = Path(__file__).parents[2]",
    "ROOT = Path('/content')"
)

# Replace any of the old model names with the new CORRECT_MODEL
for old_model in OLD_MODELS_TO_REPLACE:
    ablation_text = ablation_text.replace(old_model, CORRECT_MODEL)

ablation_path.write_text(ablation_text)
assert CORRECT_MODEL in ablation_path.read_text()
assert "Path('/content')" in ablation_path.read_text()
print(f"[OK] run_ablation.py -> ROOT=/content, model={CORRECT_MODEL}")

print("\nBoth scripts verified. Ready to train.")

[OK] train_simpo_judge.py -> unsloth/Qwen2.5-3B-Instruct
[OK] run_ablation.py -> ROOT=/content, model=unsloth/Qwen2.5-3B-Instruct

Both scripts verified. Ready to train.


In [10]:
from pathlib import Path

train_script_path = Path("/content/train_simpo_judge.py")

if train_script_path.exists():
    content = train_script_path.read_text()

    # Define target imports
    future_import = "from __future__ import annotations"
    unsloth_import = "from unsloth import FastLanguageModel"

    # Remove all instances of these imports from the content
    lines = content.splitlines()
    filtered_lines = [line for line in lines if line.strip() != future_import and line.strip() != unsloth_import]
    content_without_imports = "\n".join(filtered_lines)

    # Reconstruct content with correct order: __future__ first, then unsloth, then the rest
    new_content_parts = []
    if future_import in content: # Only add if it was originally present
        new_content_parts.append(future_import)
    if unsloth_import in content: # Only add if it was originally present
        new_content_parts.append(unsloth_import)
    new_content_parts.append(content_without_imports)
    new_content = "\n".join(new_content_parts).strip()

    train_script_path.write_text(new_content)
    print("Fixed: Ensured 'from __future__' is at the top, followed by 'from unsloth' in train_simpo_judge.py")

# --- Step 2: Fix double-2507 in both scripts ---
for f in ["/content/train_simpo_judge.py", "/content/run_ablation.py"]:
    p = Path(f)
    if p.exists():
        current_content = p.read_text()
        updated_content = current_content.replace("-2507-2507", "-2507")
        p.write_text(updated_content)
print("Fixed: Removed double '-2507' from script filenames.")

# --- Step 3: Fix dataset path in train_simpo_judge.py ---
# Re-read train_simpo_judge.py content as it might have been modified by previous steps
if train_script_path.exists():
    train_script_content = train_script_path.read_text()
    updated_content = train_script_content.replace(
        "tenacious_judge_train_v2.jsonl",
        "tenacious_judge_act5_pairs.jsonl"
    )
    train_script_path.write_text(updated_content)
    print("Fixed: Changed dataset path in train_simpo_judge.py to tenacious_judge_act5_pairs.jsonl")


print("All pre-training fixes applied! Re-running training...")
!python /content/train_simpo_judge.py

Fixed: Ensured 'from __future__' is at the top, followed by 'from unsloth' in train_simpo_judge.py
Fixed: Removed double '-2507' from script filenames.
Fixed: Changed dataset path in train_simpo_judge.py to tenacious_judge_act5_pairs.jsonl
All pre-training fixes applied! Re-running training...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Installing dependencies…
Installation complete.

GPU: Tesla T4  |  VRAM: 15.6 GB
dtype: float16

Loading unsloth/Qwen2.5-3B-Instruct…
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which 

In [11]:
!python /content/run_ablation.py \
    --adapter-path /content/outputs/tenacious_judge_adapter \
    --output-dir /content/ablations


Device   : cuda
Backbone : unsloth/Qwen2.5-3B-Instruct
Adapter  : /content/outputs/tenacious_judge_adapter
Partition: dev

Loaded 78 tasks from 'dev'

[1/3] Deterministic scorer (score_task)...
  Ground truth: PASS=43  REJECT=35
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading tokenizer from unsloth/Qwen2.5-3B-Instruct...
tokenizer_config.json: 7.36kB [00:00, 17.3MB/s]
vocab.json: 2.78MB [00:00, 61.1MB/s]
merges.txt: 1.67MB [00:00, 90.9MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:00<00:00, 18.0MB/s]
added_tokens.json: 100% 605/605 [00:00<00:00, 3.18MB/s]
special_tokens_map.json: 100% 614/614 [00:00<00:00, 3.22MB/s]
Loading base model (unsloth/Qwen2.5-3B-Instruct)...
Loading weights: 100% 434/434 [00:13<00:00, 31.34it/s]
  Loaded in 14.8s
Applying LoRA adapter from /content/outputs/tenacious_judge_adapter...

[2/3] Base model inference (78 tasks)...
  20/78
  40/78
  60/78
  78/78
  UNKNOWN outputs: 0
  Ba

In [12]:
!python /content/run_ablation.py \
    --partition held_out \
    --adapter-path /content/outputs/tenacious_judge_adapter \
    --output-dir /content/ablations


Device   : cuda
Backbone : unsloth/Qwen2.5-3B-Instruct
Adapter  : /content/outputs/tenacious_judge_adapter
Partition: held_out

Loaded 65 tasks from 'held_out'

[1/3] Deterministic scorer (score_task)...
  Ground truth: PASS=26  REJECT=39
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading tokenizer from unsloth/Qwen2.5-3B-Instruct...
Loading base model (unsloth/Qwen2.5-3B-Instruct)...
Loading weights: 100% 434/434 [00:08<00:00, 51.10it/s]
  Loaded in 9.5s
Applying LoRA adapter from /content/outputs/tenacious_judge_adapter...

[2/3] Base model inference (65 tasks)...
  20/65
  40/65
  60/65
  65/65
  UNKNOWN outputs: 0
  Base accuracy: 50.8%  (33/65)

[3/3] Trained model inference (65 tasks)...
  20/65
  40/65
  60/65
  65/65
  UNKNOWN outputs: 33

  ABLATION RESULTS   partition=held_out   n=65
  Ground truth dist  : PASS=26  REJECT=39
  (1) Deterministic  : 100.0%  (baseline -- defines ground truth)
  (2)

In [13]:
import json, shutil
from pathlib import Path
from google.colab import files

results = json.loads(Path("/content/ablations/ablation_results.json").read_text())
s = results["summary"]
m = results["metadata"]

print("=" * 60)
print(f"  ABLATION RESULTS — {m['partition']} ({m['n_tasks']} tasks)")
print("=" * 60)
print(f"  Backbone       : {m['backbone']}")
print(f"  Deterministic  : 100.0%")
print(f"  Base model     : {s['base_model']['accuracy']:.1%}  ({s['base_model']['correct']}/{s['base_model']['total']})")
print(f"  Trained judge  : {s['trained_model']['accuracy']:.1%}  ({s['trained_model']['correct']}/{s['trained_model']['total']})")
if s.get('delta_a_trained_vs_gt') is not None:
    print(f"  Delta A        : {s['delta_a_trained_vs_gt']:+.1%}")
    print(f"  Delta B        : {s['delta_b_trained_vs_base']:+.1%}")
print("=" * 60)

shutil.make_archive("/content/act_iii_iv_outputs", "zip", "/content/outputs")
shutil.make_archive("/content/ablation_results", "zip", "/content/ablations")
files.download("/content/act_iii_iv_outputs.zip")
files.download("/content/ablation_results.zip")


  ABLATION RESULTS — held_out (65 tasks)
  Backbone       : unsloth/Qwen2.5-3B-Instruct
  Deterministic  : 100.0%
  Base model     : 50.8%  (33/65)
  Trained judge  : 59.4%  (19/32)
  Delta A        : -40.6%
  Delta B        : +8.6%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>